In [1]:
import sys
sys.path.insert(0, "..")   # project root, so `src` can be imported

import time                        # timing training and prediction

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Preprocessing and pipelines
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# Models
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC

# Evaluation
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

from src.data import get_numpy_data


In [ ]:
def evaluate_model(model, name):
    start = time.perf_counter()
    model.fit(X_train, y_train)
    train_time = time.perf_counter() - start

    start = time.perf_counter()
    val_pred = model.predict(X_val)
    pred_time = time.perf_counter() - start

    val_acc = accuracy_score(y_val, val_pred)
    # prediction time in ms: fast models (e.g. logistic regression) round to 0.00 in seconds
    print(f"{name}: val acc {val_acc:.3f} | train {train_time:.1f}s | "
          f"predict {pred_time * 1000:.1f} ms ({pred_time / len(X_val) * 1000:.4f} ms/image)")
    return {"model": name, "val_acc": val_acc, "train_time": train_time, "pred_time": pred_time}

results = []   # one row per chosen model, shown as a table at the end


In [3]:
(X_train, y_train), (X_val, y_val), (X_test, y_test) = get_numpy_data()

print(X_train.shape, y_train.shape)   # (55000, 784) (55000,)
print(X_val.shape,   y_val.shape)     # (5000, 784)  (5000,)
print(X_test.shape,  y_test.shape)    # (10000, 784) (10000,)


(55000, 784) (55000,)
(5000, 784) (5000,)
(10000, 784) (10000,)


In [12]:
logreg = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(max_iter=1000)),
])


evaluate_model(logreg, "LogReg (scaled, C=1)");   # ; hides the returned dict

print("Training set score: {:.3f}".format(logreg.score(X_train, y_train)))

LogReg (scaled, C=1): val acc 0.910 | train 12.4s | predict 0.01s (0.003 ms/image)
Training set score: 0.945


In [13]:
for C in [1.0, 0.1, 0.01]:
    model = Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(C=C, max_iter=1000)),
    ])
    evaluate_model(model, f"LogReg (scaled, C={C})")
    print("Training set score: {:.3f}".format(model.score(X_train, y_train)))

LogReg (scaled, C=1.0): val acc 0.910 | train 16.6s | predict 0.01s (0.002 ms/image)
Training set score: 0.945
LogReg (scaled, C=0.1): val acc 0.916 | train 6.5s | predict 0.01s (0.002 ms/image)
Training set score: 0.944
LogReg (scaled, C=0.01): val acc 0.917 | train 6.8s | predict 0.01s (0.002 ms/image)
Training set score: 0.937


In [14]:
logreg_c_001 = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(C=0.01, max_iter=1000)),
])

results.append(evaluate_model(logreg_c_001, "LogReg (scaled, C=0.01)"))   # chosen logistic regression

print("Training set score: {:.3f}".format(logreg_c_001.score(X_train, y_train)))

LogReg (scaled, C=0.01): val acc 0.917 | train 8.1s | predict 0.01s (0.002 ms/image)
Training set score: 0.937


In [15]:
logreg_raw = LogisticRegression(max_iter=1000)

evaluate_model(logreg_raw, "LogReg (raw pixels, C=1)")

print("Training set score: {:.3f}".format(logreg_raw.score(X_train, y_train)))

LogReg (raw pixels, C=1): val acc 0.914 | train 12.1s | predict 0.00s (0.001 ms/image)
Training set score: 0.940


### Logistic regression: comparison

| Model | Scaler | C | Training time | Prediction time (5k images) | Train acc | Val acc | Train–val gap | Notes |
|---|---|---|---|---|---|---|---|---|
| `logreg_raw` | No | 1.0 | 12.5s | < 5 ms | 0.940 | 0.914 | 2.6 | Raw [0, 1] pixels; slowest to converge |
| `logreg` | StandardScaler | 1.0 | 8.5s | ~10 ms | 0.945 | 0.910 | 3.5 | Scaling inflates rare edge pixels → mild overfitting |
| C loop | StandardScaler | 0.1 | 7.8s | ~10 ms | 0.944 | 0.916 | 2.8 | More regularization, better val |
| `logreg_c_001` | StandardScaler | **0.01** | **5–6.5s** | ~10 ms | 0.937 | **0.917** | **2.0** | **Best:** highest val acc, smallest gap, fastest to train |

Times vary by a second or two between runs; accuracies are identical every run (the solver is deterministic).
Prediction is a single matrix multiplication, so it takes milliseconds; the scaled versions are slightly slower
because the scaler transforms the 5k images first.

**Choice:** StandardScaler + `C=0.01`, about 91.7% validation accuracy. This is the linear baseline the other models have to beat.


In [8]:
knn = KNeighborsClassifier()   # default k=5

evaluate_model(knn, "k-NN (k=5)");

k-NN (k=5): val acc 0.971 | train 0.0s | predict 1.02s (0.205 ms/image)


In [9]:
for K in [3, 5, 7, 9, 11]:
    evaluate_model(KNeighborsClassifier(n_neighbors=K), f"k-NN (k={K})")

k-NN (k=3): val acc 0.976 | train 0.0s | predict 0.85s (0.170 ms/image)
k-NN (k=5): val acc 0.971 | train 0.0s | predict 0.85s (0.170 ms/image)
k-NN (k=7): val acc 0.969 | train 0.0s | predict 0.93s (0.185 ms/image)
k-NN (k=9): val acc 0.969 | train 0.0s | predict 0.86s (0.173 ms/image)
k-NN (k=11): val acc 0.967 | train 0.0s | predict 0.87s (0.175 ms/image)


In [10]:
knn_3 = KNeighborsClassifier(n_neighbors=3)

results.append(evaluate_model(knn_3, "k-NN (k=3)"))   # chosen k-NN

k-NN (k=3): val acc 0.976 | train 0.0s | predict 0.89s (0.177 ms/image)


In [16]:
for n in [30, 50, 100]:
    model = Pipeline([("pca", PCA(n_components=n, random_state=42)),
                      ("knn", KNeighborsClassifier(n_neighbors=3))])
    evaluate_model(model, f"PCA({n}) + k-NN (k=3)")

PCA(30) + k-NN (k=3): val acc 0.975 | train 0.4s | predict 0.12s (0.024 ms/image)
PCA(50) + k-NN (k=3): val acc 0.977 | train 0.4s | predict 0.13s (0.026 ms/image)
PCA(100) + k-NN (k=3): val acc 0.977 | train 0.4s | predict 0.16s (0.032 ms/image)


In [11]:
# Comparison of the chosen models (validation set)
pd.DataFrame(results).sort_values("val_acc", ascending=False).round(3)

,model,val_acc,train_time,pred_time
1,k-NN (k=3),0.976,0.021,0.887
0,"LogReg (scaled, C=0.01)",0.917,6.513,0.013
